# Speech transcription using Whipser model

This project demonstrates an end‑to‑end audio processing pipeline that converts speech to text using OpenAI Whisper and then generates concise summaries using BART (facebook/bart-large-cnn). A Gradio web interface is provided for easy interaction

Features

* Automatic Speech Recognition (ASR) with Whisper

* Multi‑language support: auto, en, fr, ar, es

* Text summarization with BART‑Large‑CNN

* Interactive Gradio UI (microphone + audio upload)



## Set  up and imports

In [14]:
!pip install torch transformers
!pip install openai-whisper
!pip install gradio
!pip install soundfile

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


In [15]:
import whisper
import torch
from transformers import pipeline
import warnings
import gradio as gr
warnings.filterwarnings('ignore')

## Whisper model loading

In [16]:
def load_whisper_model(model_size="base"):
    """Load Whisper model for transcription"""
    print(f"Loading Whisper {model_size} model...")
    model = whisper.load_model(model_size)
    print(f" Whisper model loaded successfully")
    return model

In [17]:
whisper_model = load_whisper_model("base")

Loading Whisper base model...
 Whisper model loaded successfully


## Summarization model loading

In [18]:
print("Loading summarization model...")
summarizer = pipeline("summarization", model="facebook/bart-large-cnn")
print("Summarizer loaded")

Loading summarization model...


Device set to use cpu


Summarizer loaded


## Transcription Function

In [19]:
def transcribe_audio(audio_path, language="en"):
    """
    Transcribe audio file to text
    
    Args:
        audio_path: path to audio file
        language: language code (en, ar, fr, es, etc.) or "auto"
    
    Returns:
        transcribed text
    """
    print(f"🎤 Transcribing: {audio_path}")
    
    result = whisper_model.transcribe(
        audio_path,
        language=None if language == "auto" else language
    )
    
    text = result["text"]
    print(f"✅ Transcription complete: {len(text)} characters")
    return text


## Summarization function

In [20]:
def summarize_text(text, max_length=130, min_length=30):
    """
    Summarize text using AI
    
    Args:
        text: text to summarize
        max_length: maximum summary length
        min_length: minimum summary length
    
    Returns:
        summary text
    """
    print("Generating summary...")
    
    max_chunk = 1024
    if len(text) > max_chunk:
        
        text = text[:max_chunk]
    
    summary = summarizer(text, max_length=max_length, min_length=min_length, do_sample=False)
    summary_text = summary[0]['summary_text']
    
    print(f"Summary complete: {len(summary_text)} characters")
    return summary_text

## Complete Pipeline

In [21]:
def transcribe_and_summarize(audio_path, language="en"):
    """
    Complete pipeline: Audio → Transcription → Summary
    
    Args:
        audio_path: path to audio file
        language: language code
    
    Returns:
        dict with transcription and summary
    """
    print("="*60)
    print("🎯 AUDIO TRANSCRIPTION & SUMMARIZATION")
    print("="*60)
    
    # Step 1: Transcribe
    transcription = transcribe_audio(audio_path, language)
    
    # Step 2: Summarize
    summary = summarize_text(transcription)
    
    # Results
    results = {
        "audio_file": audio_path,
        "transcription": transcription,
        "summary": summary,
        "word_count": len(transcription.split())
    }
    
    print("\n" + "="*60)
    print("📄 TRANSCRIPTION:")
    print("="*60)
    print(transcription)
    print("\n" + "="*60)
    print("📋 SUMMARY:")
    print("="*60)
    print(summary)
    print("="*60)
    
    return results

# Gradio Interface

## Backend Function

In [22]:
def gradio_pipeline(audio, language):
    """
    Gradio wrapper for transcription + summarization
    """
    if audio is None:
        return "No audio provided.", ""

    transcription = whisper_model.transcribe(
        audio,
        language=None if language == "auto" else language
    )["text"]

    max_chunk = 1024
    text = transcription[:max_chunk]

    summary = summarizer(
        text,
        max_length=130,
        min_length=30,
        do_sample=False
    )[0]["summary_text"]

    return transcription, summary


## Gardio UI definition

In [23]:
interface = gr.Interface(
    fn=gradio_pipeline,
    inputs=[
        gr.Audio(
            sources=["microphone", "upload"],
            type="filepath",
            label="🎙️ Record or Upload Audio"
        ),
        gr.Dropdown(
            choices=["auto", "en", "fr", "ar", "es"],
            value="auto",
            label="🌍 Language"
        )
    ],
    outputs=[
        gr.Textbox(label="📄 Transcription", lines=10),
        gr.Textbox(label="📋 Summary", lines=5)
    ],
    title="Speech Transcription & Summarization",
    description="Whisper (ASR) + BART (Summarization) — Kaggle Compatible"
)


## Launch application

In [ ]:
interface.launch(
    share=True,
    debug=True,
    inline=True
)


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8ef6077ba7ed5a3dbd.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


Your max_length is set to 130, but your input_length is only 91. Since this is a summarization task, where outputs shorter than the input are typically wanted, you might consider decreasing max_length manually, e.g. summarizer('...', max_length=45)
